# 11. Cleanup and final selection layer

Limpia referencias obsoletas, consolida decisiones por epoch y crea la capa final `coregistration_selected/` lista para DoD.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys

REPO_PATH = Path('/content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo')
SRC_PATH = REPO_PATH / 'src'

assert REPO_PATH.exists(), f'No existe: {REPO_PATH}'
assert SRC_PATH.exists(), f'No existe: {SRC_PATH}'

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

print('REPO_PATH:', REPO_PATH)
print('SRC_PATH :', SRC_PATH)

REPO_PATH: /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo
SRC_PATH : /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/src


In [3]:
import json
import shutil
from pathlib import Path
import pandas as pd
import numpy as np

## Configuración

In [4]:
BASE_EPOCH = '2025_B'

QA_DIR = REPO_PATH / 'outputs' / 'qa'
COREG_DIR = REPO_PATH / 'outputs' / 'coregistration_nk'
DIAG_DIR = REPO_PATH / 'outputs' / 'coregistration_diagnostics'
FINAL_DIR = REPO_PATH / 'outputs' / 'coregistration_final'
SELECTED_DIR = REPO_PATH / 'outputs' / 'coregistration_selected'
LEGACY_DIR = REPO_PATH / 'outputs' / '_legacy'
WEB_DIR = REPO_PATH / 'web'
WEB_TABLES_DIR = WEB_DIR / 'tables'

for p in [FINAL_DIR, SELECTED_DIR, LEGACY_DIR, WEB_TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

inventory_csv = QA_DIR / 'project_raster_inventory.csv'
scenario_csv = DIAG_DIR / 'coreg_diag_scenario_comparison.csv'
decision_csv = DIAG_DIR / 'coreg_diag_decision_table.csv'
nk_results_csv = COREG_DIR / 'coregistration_nk_results.csv'

assert inventory_csv.exists(), f'No existe: {inventory_csv}'
assert scenario_csv.exists(), f'No existe: {scenario_csv}'
assert nk_results_csv.exists(), f'No existe: {nk_results_csv}'

print('inventory_csv :', inventory_csv)
print('scenario_csv  :', scenario_csv)
print('decision_csv  :', decision_csv if decision_csv.exists() else 'No existe aún')
print('nk_results_csv:', nk_results_csv)
print('SELECTED_DIR  :', SELECTED_DIR)

inventory_csv : /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/qa/project_raster_inventory.csv
scenario_csv  : /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/coregistration_diagnostics/coreg_diag_scenario_comparison.csv
decision_csv  : /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/coregistration_diagnostics/coreg_diag_decision_table.csv
nk_results_csv: /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/coregistration_nk/coregistration_nk_results.csv
SELECTED_DIR  : /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/coregistration_selected


## Carga de datos

In [5]:
inventory_df = pd.read_csv(inventory_csv)
scenario_df = pd.read_csv(scenario_csv)
nk_df = pd.read_csv(nk_results_csv)
decision_df = pd.read_csv(decision_csv) if decision_csv.exists() else None

print('inventory_df:', len(inventory_df))
print('scenario_df :', len(scenario_df))
print('nk_df       :', len(nk_df))
print('decision_df :', len(decision_df) if decision_df is not None else 0)

display(inventory_df.head())
display(scenario_df.head())
display(nk_df.head())
if decision_df is not None:
    display(decision_df.head())

inventory_df: 12
scenario_df : 60
nk_df       : 5
decision_df : 5


,filename,epoch,product_type,source_path,path,exists,opened,error,crs,width,...,bounds_top,driver,valid_name,qa_exists,qa_opened,qa_has_crs,qa_has_dimensions,qa_has_bands,qa_valid_name,qa_ok
0,2017_DSM.tif,2017,dsm,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,True,NaN,EPSG:32719,8991,...,8225448.0,GTiff,True,True,True,True,True,True,True,True
1,2017_Ortho.tif,2017,ortho,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,True,NaN,EPSG:32719,8991,...,8225448.0,GTiff,True,True,True,True,True,True,True,True
2,2018_DSM.tif,2018,dsm,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,True,NaN,EPSG:32719,13010,...,8225539.0,GTiff,True,True,True,True,True,True,True,True
3,2018_Ortho.tif,2018,ortho,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,True,NaN,EPSG:32719,13010,...,8225539.0,GTiff,True,True,True,True,True,True,True,True
4,2020_DSM.tif,2020,dsm,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,True,NaN,EPSG:32719,2506,...,8225602.0,GTiff,True,True,True,True,True,True,True,True


,scenario,n_before,n_after,mean_before,mean_after,median_before,median_after,std_before,std_after,nmad_before,...,mean_abs_before,mean_abs_after,median_abs_before,median_abs_after,mean_improved,median_improved,std_improved,nmad_improved,target_epoch,base_epoch
0,stable_raw_raw,1766731,1766731,5.579150,3.371101,3.594238,1.130371,5.445805,5.271004,4.517297,...,5.886564,4.313142,3.757812,2.097168,True,True,True,True,2017,2025_B
1,stable_raw_clip1_99,1731399,1731401,5.526326,3.306667,3.594238,1.130371,5.122027,4.963112,4.385542,...,5.726471,4.134896,3.629395,2.000977,True,True,True,True,2017,2025_B
2,stable_raw_clip5_95,1590081,1590074,5.347439,3.103891,3.594238,1.130371,4.455482,4.313270,3.916438,...,5.380257,3.678666,3.594238,1.678711,True,True,True,True,2017,2025_B
3,stable_no_border_raw,1766731,1766731,5.579150,3.371101,3.594238,1.130371,5.445805,5.271004,4.517297,...,5.886564,4.313142,3.757812,2.097168,True,True,True,True,2017,2025_B
4,stable_no_border_clip1_99,1731399,1731401,5.526326,3.306667,3.594238,1.130371,5.122027,4.963112,4.385542,...,5.726471,4.134896,3.629395,2.000977,True,True,True,True,2017,2025_B


,base_dem_path,target_dem_path,mask_vector_path,output_path,factor_giro,poly_order,status,epoch,base_epoch
0,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,-0.05,1,ok,2017,2025_B
1,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,-0.05,1,ok,2018,2025_B
2,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,-0.05,1,ok,2020,2025_B
3,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,-0.05,1,ok,2023,2025_B
4,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,-0.05,1,ok,2025_A,2025_B


,target_epoch,best_raw_scenario,best_raw_nmad_after,best_raw_median_abs_after,best_clip1_99_scenario,best_clip1_99_nmad_after,best_clip1_99_median_abs_after,outlier_sensitivity_mean_after,raw_mean_after,clip1_99_mean_after
0,2017,stable_raw_raw,3.681163,2.097168,stable_raw_clip1_99,3.555200,2.000977,0.064434,3.371101,3.306667
1,2018,stable_raw_raw,5.250634,3.856445,stable_raw_clip1_99,5.105125,3.746094,0.057351,-2.107840,-2.165191
2,2020,stable_no_border_raw,6.162418,4.173340,stable_no_border_clip1_99,5.968044,4.028320,0.039816,-0.908376,-0.948192
3,2023,stable_no_border_raw,0.323595,0.201660,stable_no_border_clip1_99,0.311288,0.193359,0.044931,1.288092,1.243161
4,2025_A,stable_raw_raw,0.097006,0.062988,stable_raw_clip1_99,0.093386,0.060547,0.000001,0.114644,0.114643


## Helpers

In [6]:
def get_original_dsm_path(epoch):
    sub = inventory_df[
        (inventory_df['epoch'].astype(str).str.strip() == str(epoch).strip()) &
        (inventory_df['product_type'].astype(str).str.lower() == 'dsm') &
        (inventory_df['qa_ok'] == True)
    ].copy()
    assert len(sub) == 1, f'No se encontró un DSM original único para epoch={epoch}'
    return Path(sub.iloc[0]['source_path'])

def get_corrected_dsm_path(epoch, base_epoch=BASE_EPOCH):
    sub = nk_df.copy()
    sub['epoch'] = sub['epoch'].astype(str).str.strip()
    sub['base_epoch'] = sub['base_epoch'].astype(str).str.strip()
    sub['status'] = sub['status'].astype(str).str.strip()

    sub = sub[
        (sub['epoch'] == str(epoch).strip()) &
        (sub['base_epoch'] == str(base_epoch).strip()) &
        (sub['status'] == 'ok')
    ].copy()

    assert len(sub) > 0, f'No se encontró DEM corregido para epoch={epoch}, base_epoch={base_epoch}'
    return Path(sub.iloc[-1]['output_path'])

def choose_best_raw_scenario(epoch):
    sub = scenario_df[scenario_df['target_epoch'].astype(str) == str(epoch)].copy()
    sub = sub[sub['scenario'].astype(str).str.endswith('_raw')].copy()
    assert len(sub) > 0, f'No hay escenarios raw para epoch={epoch}'
    sub = sub.sort_values(
        by=['nmad_after', 'median_abs_after', 'mean_abs_after', 'std_after'],
        ascending=[True, True, True, True]
    ).reset_index(drop=True)
    return sub.iloc[0].to_dict()

def classify_decision(row):
    mean_improved = bool(row['mean_improved'])
    median_improved = bool(row['median_improved'])
    std_improved = bool(row['std_improved'])
    nmad_improved = bool(row['nmad_improved'])
    mean_after = float(row['mean_after'])
    median_after = float(row['median_after'])

    if (not median_improved) and (not mean_improved):
        return 'use_original'
    if median_improved and nmad_improved and std_improved and abs(mean_after) <= 3.0:
        return 'use_corrected'
    if median_improved and nmad_improved and abs(mean_after) <= 5.0:
        return 'use_corrected'
    if median_improved and abs(median_after) <= 0.25 and abs(mean_after) <= 2.0:
        return 'use_corrected_nearly_neutral'
    return 'review'

def selected_product_from_decision(decision):
    if decision in ['use_corrected', 'use_corrected_nearly_neutral']:
        return 'corrected'
    if decision == 'use_original':
        return 'original'
    return 'review'

def copy_selected(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return dst

def rationale(decision):
    return {
        'use_corrected': 'La corrección mejora ajuste central y robusto.',
        'use_corrected_nearly_neutral': 'La corrección tiene efecto pequeño y no perjudica el ajuste.',
        'use_original': 'La corrección empeora el ajuste central; se conserva el original.',
        'review': 'El resultado es mixto y debe revisarse antes de usar en DoD.'
    }[decision]

## Capa final de selección

In [7]:
epochs = sorted(scenario_df['target_epoch'].astype(str).unique().tolist())

selection_records = []

for epoch in epochs:
    best = choose_best_raw_scenario(epoch)
    decision = classify_decision(best)
    selected_product = selected_product_from_decision(decision)

    original_path = get_original_dsm_path(epoch)
    corrected_path = get_corrected_dsm_path(epoch)

    if selected_product == 'corrected':
        chosen_src = corrected_path
    elif selected_product == 'original':
        chosen_src = original_path
    else:
        chosen_src = None

    if chosen_src is not None:
        out_name = f'{epoch}_DSM_selected.tif'
        selected_path = SELECTED_DIR / out_name
        copy_selected(chosen_src, selected_path)
        selected_path_str = str(selected_path)
    else:
        selected_path_str = None

    selection_records.append({
        'epoch': epoch,
        'base_epoch': BASE_EPOCH,
        'best_raw_scenario': best['scenario'],
        'mean_after': best['mean_after'],
        'median_after': best['median_after'],
        'std_after': best['std_after'],
        'nmad_after': best['nmad_after'],
        'mean_improved': bool(best['mean_improved']),
        'median_improved': bool(best['median_improved']),
        'std_improved': bool(best['std_improved']),
        'nmad_improved': bool(best['nmad_improved']),
        'decision': decision,
        'selected_product': selected_product,
        'decision_rationale': rationale(decision),
        'original_path': str(original_path),
        'corrected_path': str(corrected_path),
        'selected_path': selected_path_str,
        'ready_for_dod': selected_product in ['corrected', 'original'],
    })

selection_df = pd.DataFrame(selection_records).sort_values('epoch').reset_index(drop=True)
display(selection_df)

,epoch,base_epoch,best_raw_scenario,mean_after,median_after,std_after,nmad_after,mean_improved,median_improved,std_improved,nmad_improved,decision,selected_product,decision_rationale,original_path,corrected_path,selected_path,ready_for_dod
0,2017,2025_B,stable_raw_raw,3.371101,1.130371,5.271004,3.681163,True,True,True,True,use_corrected,corrected,La corrección mejora ajuste central y robusto.,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
1,2018,2025_B,stable_raw_raw,-2.107840,-2.006836,5.271650,5.250634,True,True,False,False,review,review,El resultado es mixto y debe revisarse antes d...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,None,False
2,2020,2025_B,stable_no_border_raw,-0.908376,-0.716797,5.858963,6.162418,True,True,True,True,use_corrected,corrected,La corrección mejora ajuste central y robusto.,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
3,2023,2025_B,stable_no_border_raw,1.288092,0.092285,2.310035,0.323595,True,True,True,False,use_corrected_nearly_neutral,corrected,La corrección tiene efecto pequeño y no perjud...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
4,2025_A,2025_B,stable_no_border_raw,0.115684,0.036621,0.443959,0.097006,False,False,True,False,use_original,original,La corrección empeora el ajuste central; se co...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True


## Base epoch: capa final

In [8]:
base_original_path = get_original_dsm_path(BASE_EPOCH)
base_selected_path = SELECTED_DIR / f'{BASE_EPOCH}_DSM_selected.tif'
copy_selected(base_original_path, base_selected_path)

base_record = pd.DataFrame([{
    'epoch': BASE_EPOCH,
    'base_epoch': BASE_EPOCH,
    'best_raw_scenario': 'base_reference',
    'mean_after': 0.0,
    'median_after': 0.0,
    'std_after': 0.0,
    'nmad_after': 0.0,
    'mean_improved': True,
    'median_improved': True,
    'std_improved': True,
    'nmad_improved': True,
    'decision': 'base_reference',
    'selected_product': 'original',
    'decision_rationale': 'DEM base de referencia para todo el proyecto.',
    'original_path': str(base_original_path),
    'corrected_path': '',
    'selected_path': str(base_selected_path),
    'ready_for_dod': True,
}])

selection_with_base_df = pd.concat([selection_df, base_record], ignore_index=True)
selection_with_base_df = selection_with_base_df.sort_values('epoch').reset_index(drop=True)
display(selection_with_base_df)

,epoch,base_epoch,best_raw_scenario,mean_after,median_after,std_after,nmad_after,mean_improved,median_improved,std_improved,nmad_improved,decision,selected_product,decision_rationale,original_path,corrected_path,selected_path,ready_for_dod
0,2017,2025_B,stable_raw_raw,3.371101,1.130371,5.271004,3.681163,True,True,True,True,use_corrected,corrected,La corrección mejora ajuste central y robusto.,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
1,2018,2025_B,stable_raw_raw,-2.107840,-2.006836,5.271650,5.250634,True,True,False,False,review,review,El resultado es mixto y debe revisarse antes d...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,None,False
2,2020,2025_B,stable_no_border_raw,-0.908376,-0.716797,5.858963,6.162418,True,True,True,True,use_corrected,corrected,La corrección mejora ajuste central y robusto.,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
3,2023,2025_B,stable_no_border_raw,1.288092,0.092285,2.310035,0.323595,True,True,True,False,use_corrected_nearly_neutral,corrected,La corrección tiene efecto pequeño y no perjud...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
4,2025_A,2025_B,stable_no_border_raw,0.115684,0.036621,0.443959,0.097006,False,False,True,False,use_original,original,La corrección empeora el ajuste central; se co...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
5,2025_B,2025_B,base_reference,0.000000,0.000000,0.000000,0.000000,True,True,True,True,base_reference,original,DEM base de referencia para todo el proyecto.,/content/drive/MyDrive/geo_projects/llojeta-gl...,,/content/drive/MyDrive/geo_projects/llojeta-gl...,True


## Limpieza lógica y marcación de legado

In [9]:
legacy_candidates = [
    REPO_PATH / 'outputs' / 'coregistration_evaluation',
    REPO_PATH / 'outputs' / 'coregistration_evaluation_batch',
    REPO_PATH / 'web' / 'assets' / 'coreg_2017_nk_corrected.json',
    REPO_PATH / 'web' / 'assets' / 'coreg_2017_nk_corrected.png',
    REPO_PATH / 'web' / 'maps' / 'coreg_2017_nk_corrected.html',
]

legacy_records = []
for p in legacy_candidates:
    legacy_records.append({
        'path': str(p),
        'exists': p.exists(),
        'kind': 'directory' if p.is_dir() else 'file' if p.exists() else 'missing',
        'action': 'mark_as_legacy'
    })

legacy_df = pd.DataFrame(legacy_records)
display(legacy_df)

,path,exists,kind,action
0,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,directory,mark_as_legacy
1,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,directory,mark_as_legacy
2,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,file,mark_as_legacy
3,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,file,mark_as_legacy
4,/content/drive/MyDrive/geo_projects/llojeta-gl...,True,file,mark_as_legacy


## Exportación de la capa final

In [10]:
selection_csv = FINAL_DIR / 'selection_manifest.csv'
selection_json = FINAL_DIR / 'selection_manifest.json'
dod_ready_csv = FINAL_DIR / 'dod_ready_epochs.csv'
legacy_csv = FINAL_DIR / 'legacy_inventory.csv'

selection_with_base_df.to_csv(selection_csv, index=False, encoding='utf-8')
selection_with_base_df.to_json(selection_json, orient='records', force_ascii=False, indent=2)

selection_with_base_df[selection_with_base_df['ready_for_dod'] == True].to_csv(
    dod_ready_csv, index=False, encoding='utf-8'
)

legacy_df.to_csv(legacy_csv, index=False, encoding='utf-8')

print('Guardado:', selection_csv)
print('Guardado:', selection_json)
print('Guardado:', dod_ready_csv)
print('Guardado:', legacy_csv)

Guardado: /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/coregistration_final/selection_manifest.csv
Guardado: /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/coregistration_final/selection_manifest.json
Guardado: /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/coregistration_final/dod_ready_epochs.csv
Guardado: /content/drive/MyDrive/geo_projects/llojeta-glacier-analysis/repo/outputs/coregistration_final/legacy_inventory.csv


## Árbol de la capa final

In [11]:
for p in sorted(SELECTED_DIR.glob('*')):
    if p.is_file():
        print(f'├── {p.name} ({p.stat().st_size / (1024*1024):.1f}MB)')
    else:
        print(f'├── {p.name}/')

├── 2017_DSM_selected.tif (86.0MB)
├── 2020_DSM_selected.tif (12.1MB)
├── 2023_DSM_selected.tif (7.3MB)
├── 2025_A_DSM_selected.tif (16.0MB)
├── 2025_B_DSM_selected.tif (30.0MB)


## Salidas que debemos revisar antes de pasar a DoD

1. `selection_with_base_df`
   - confirmar qué epoch usa `corrected` y cuál `original`
2. `legacy_df`
   - confirmar qué quedará como histórico/obsoleto
3. árbol de `coregistration_selected/`
   - verificar que todos los TIFF seleccionados existen
4. `dod_ready_epochs.csv`
   - esta será la entrada directa al notebook de DoD

In [12]:
print('=== RESUMEN FINAL ===')
print('Base epoch:', BASE_EPOCH)
print('Decisions:', selection_df['decision'].value_counts().to_dict())
print('Selected products:', selection_df['selected_product'].value_counts().to_dict())
print('Ready for DoD:', int(selection_with_base_df['ready_for_dod'].sum()))
display(selection_with_base_df)

=== RESUMEN FINAL ===
Base epoch: 2025_B
Decisions: {'use_corrected': 2, 'review': 1, 'use_corrected_nearly_neutral': 1, 'use_original': 1}
Selected products: {'corrected': 3, 'review': 1, 'original': 1}
Ready for DoD: 5


,epoch,base_epoch,best_raw_scenario,mean_after,median_after,std_after,nmad_after,mean_improved,median_improved,std_improved,nmad_improved,decision,selected_product,decision_rationale,original_path,corrected_path,selected_path,ready_for_dod
0,2017,2025_B,stable_raw_raw,3.371101,1.130371,5.271004,3.681163,True,True,True,True,use_corrected,corrected,La corrección mejora ajuste central y robusto.,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
1,2018,2025_B,stable_raw_raw,-2.107840,-2.006836,5.271650,5.250634,True,True,False,False,review,review,El resultado es mixto y debe revisarse antes d...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,None,False
2,2020,2025_B,stable_no_border_raw,-0.908376,-0.716797,5.858963,6.162418,True,True,True,True,use_corrected,corrected,La corrección mejora ajuste central y robusto.,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
3,2023,2025_B,stable_no_border_raw,1.288092,0.092285,2.310035,0.323595,True,True,True,False,use_corrected_nearly_neutral,corrected,La corrección tiene efecto pequeño y no perjud...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
4,2025_A,2025_B,stable_no_border_raw,0.115684,0.036621,0.443959,0.097006,False,False,True,False,use_original,original,La corrección empeora el ajuste central; se co...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
5,2025_B,2025_B,base_reference,0.000000,0.000000,0.000000,0.000000,True,True,True,True,base_reference,original,DEM base de referencia para todo el proyecto.,/content/drive/MyDrive/geo_projects/llojeta-gl...,,/content/drive/MyDrive/geo_projects/llojeta-gl...,True
